# 03 — Small-molecule binding: pocket chemistry and topology
Audit-driven: W/C/H/Y/F/M enrichment, ACCESS/TOPO signal → SM_* blocks + residue↔pocket hypergraph.

Dev-only contract: every feature / model choice uses DevSet1010 official 5-fold CV;
TestSet300 / TestSetNew46 are opened only after the locks. Each call below reuses its
Drive cache (features, pocket banks, fold checkpoints, predictions) and runs missing upstream stages itself.

In [ ]:
# ============================================================
# BOOTSTRAP — Colab: mount Drive, put pbind on the path
# ============================================================
import sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Folder of this repo (copy it to Drive once, or git clone it into /content).
REPO = Path("/content/drive/MyDrive/Protein-binding-Phi/protein-binding")
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

# Data root (Drive layout). Override with PBIND_BASE for a local mirror.
# import os; os.environ["PBIND_BASE"] = "/content/drive/MyDrive/Protein-binding-Phi"

from pbind.paths import resolve_paths, check_inputs
display(check_inputs(resolve_paths()))

In [ ]:
from pbind.pipelines import open_workspace, features, screens, v1, v2, final

ws = open_workspace("small")          # splits + official folds + GEAR context (+EVO/ESMIF if cached)
ws.task.label, sorted(ws.available)

## Part A — structures + task-specific feature blocks

In [ ]:
features.ensure_structures(ws)                 # shared exact chain-A PDB cache (Drive tar.gz)
display(features.build_features(ws))            # rebuilt only if a protein's cache is missing/invalid
display(features.feature_audit(ws))

## Part B — same-MLP mechanism discovery (Dev only)

In [ ]:
display(screens.family_screen(ws))
ladder, rep = screens.mechanism_ladder(ws)
display(ladder); rep

## Part C — multimodal fusion architectures

In [ ]:
arch = v1.architecture_screen(ws)
display(arch["summary"]); arch["lock"]

## Part D — pocket hypergraph

In [ ]:
hg = v1.hypergraph_screen(ws)
display(hg["summary"]); hg["lock"]

## Part E–G — global Dev lock, external evaluation, baselines, insight

In [ ]:
v1.global_lock(ws)
display(v1.external_evaluation(ws))
B = v1.baselines(ws); display(B["baselines"])
v1.insight_summary(ws)

## V2 — matched pocket-message ablation (protein-balanced BCE ± hard negatives)

In [ ]:
audit, per_protein = v2.pocket_coverage_audit(ws)
display(audit)
res = v2.v2_ablation(ws)
display(res["summary"]); display(res["ablation"]); res["locked"]

In [ ]:
display(v2.v2_external(ws))

## Final — V1 attention on the multi-scale pocket bank, Dev-locked ensemble, paper-ready report

In [ ]:
final.multiscale_screen(ws)
display(final.multiscale_external(ws))
paper = final.paper_report(ws)
display(paper["headline"]); display(paper["final"]); paper["decision"]

In [ ]:
v1.manifest(ws)